# Waypoint Property & Casualty -- Exploratory Data Analysis
**Project:** waypoint-retention-analytics
**Author:** Luciano Casillas
**Date:** 2026-05-02

Exploratory analysis of 90,000 synthetic auto and renters policies (2023-2025).
Covers data quality, target variable distribution, feature distributions, and
key bivariate relationships that motivate the predictive model in Notebook 2.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Locate project root
def find_project_root():
    p = Path.cwd()
    while p != p.parent:
        if (p / "PROJECT_MANIFEST.json").exists():
            return p
        p = p.parent
    return Path(__file__).resolve().parents[4]

PROJECT_ROOT = find_project_root()
DATA_PATH    = PROJECT_ROOT / "_build" / "workflows" / "02_data_generation" / "outputs" / "waypoint_retention.csv"

plt.rcParams.update({"figure.dpi": 100, "font.size": 11})
BLUE   = "#0077B3"
NAVY   = "#003366"
ORANGE = "#E07B00"
STEEL  = "#A0B4C8"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path:    {DATA_PATH}")

## 1. Dataset Overview

In [ ]:
df = pd.read_csv(DATA_PATH)
df["claim_severity_band"] = df["claim_severity_band"].fillna("None")
df["effective_date"] = pd.to_datetime(df["effective_date"])

print(f"Shape:   {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Null values:  {df.isnull().sum().sum()}")
print()
print("Dtypes:")
print(df.dtypes.value_counts())
print()
print("Date range:", df["effective_date"].min().date(), "to", df["effective_date"].max().date())
print("Cohort quarters:", df["cohort_quarter"].nunique(), "distinct")
print("States:", df["state"].nunique())
print("Unique customers:", df["customer_id"].nunique():,)
print("Unique policies: ", df["policy_id"].nunique():,)

In [ ]:
# Target variable summary
labeled = df[df["renewed"].isin([0, 1])]
unknown = df[df["renewed"] == -1]

print("=== Target Variable: renewed ===")
print(f"Labeled (known outcome):  {len(labeled):,}  ({len(labeled)/len(df)*100:.1f}%)")
print(f"Unknown (-1 sentinel):    {len(unknown):,}  ({len(unknown)/len(df)*100:.1f}%)")
print()
print(f"Renewal rate (labeled):   {labeled['renewed'].mean():.1%}")
print(f"Lapse rate (labeled):     {1 - labeled['renewed'].mean():.1%}")

## 2. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Overall renewal rate
rates = labeled["renewed"].value_counts(normalize=True).sort_index()
axes[0].bar(["Lapsed (0)", "Renewed (1)"], rates.values * 100, color=[ORANGE, BLUE], width=0.5)
axes[0].set_title("12-Month Renewal Outcome Distribution", fontsize=12)
axes[0].set_ylabel("% of Labeled Policies")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(rates.values):
    axes[0].text(i, v * 100 + 0.5, f"{v:.1%}", ha="center", fontsize=11, fontweight="bold")

# Renewal rate by cohort quarter
cq = labeled.groupby("cohort_quarter")["renewed"].mean().reset_index()
axes[1].plot(cq["cohort_quarter"], cq["renewed"] * 100, color=BLUE, marker="o", linewidth=2)
axes[1].axhline(labeled["renewed"].mean() * 100, color=STEEL, linestyle="--", label=f"Book avg {labeled['renewed'].mean():.1%}")
axes[1].set_title("12-Month Renewal Rate by Cohort Quarter", fontsize=12)
axes[1].set_ylabel("Renewal Rate (%)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Annual premium
axes[0,0].hist(df["annual_premium"], bins=40, color=BLUE, edgecolor="white")
axes[0,0].set_title("Annual Premium Distribution")
axes[0,0].set_xlabel("Premium ($)")
axes[0,0].axvline(df["annual_premium"].mean(), color=ORANGE, linestyle="--", label=f"Mean ${df['annual_premium'].mean():,.0f}")
axes[0,0].legend()

# Coverage tier
tier_order = ["Liability Only", "Standard", "Premium", "Elite"]
tier_counts = df["coverage_tier"].value_counts().reindex(tier_order)
axes[0,1].bar(tier_counts.index, tier_counts.values / len(df) * 100, color=BLUE)
axes[0,1].set_title("Coverage Tier Distribution")
axes[0,1].set_ylabel("% of Policies")
axes[0,1].tick_params(axis="x", rotation=15)
axes[0,1].yaxis.set_major_formatter(mtick.PercentFormatter())

# Acquisition channel
ch_counts = df["acquisition_channel"].value_counts()
axes[0,2].barh(ch_counts.index, ch_counts.values / len(df) * 100, color=BLUE)
axes[0,2].set_title("Acquisition Channel")
axes[0,2].set_xlabel("% of Policies")
axes[0,2].xaxis.set_major_formatter(mtick.PercentFormatter())

# Payment consistency score
axes[1,0].hist(df["payment_consistency"], bins=30, color=STEEL, edgecolor="white")
axes[1,0].set_title("Payment Consistency Score")
axes[1,0].set_xlabel("Score (0-100)")

# Missed payment count
mp = df["missed_payment_count"].value_counts().sort_index().head(6)
axes[1,1].bar(mp.index.astype(str), mp.values / len(df) * 100, color=BLUE)
axes[1,1].set_title("Missed Payment Count (12mo)")
axes[1,1].set_ylabel("% of Policies")
axes[1,1].set_xlabel("Count")
axes[1,1].yaxis.set_major_formatter(mtick.PercentFormatter())

# Claims distribution
axes[1,2].bar(["No Claim", "Has Claim"],
    [1 - df["has_claim_12mo"].mean(), df["has_claim_12mo"].mean()],
    color=[STEEL, BLUE])
axes[1,2].set_title("Claims in First 12 Months")
axes[1,2].set_ylabel("% of Policies")
axes[1,2].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

print(f"Avg premium:         ${df['annual_premium'].mean():,.2f}")
print(f"Multi-product rate:  {df['multi_product_flag'].mean():.1%}")
print(f"Claim rate:          {df['has_claim_12mo'].mean():.1%}")
print(f"NSF rate:            {df['nsf_flag'].mean():.1%}")

## 4. Key Bivariate Relationships: Renewal Rate by Feature

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
labeled_df = df[df["renewed"].isin([0, 1])].copy()
book_avg = labeled_df["renewed"].mean()

def bar_renewal(ax, col, title, order=None, rotation=0):
    rates = labeled_df.groupby(col)["renewed"].mean().sort_values(ascending=False)
    if order:
        rates = rates.reindex(order)
    colors = [BLUE if v == rates.max() else STEEL for v in rates.values]
    ax.bar(rates.index, rates.values * 100, color=colors)
    ax.axhline(book_avg * 100, color=ORANGE, linestyle="--", label=f"Book avg {book_avg:.1%}")
    ax.set_title(title)
    ax.set_ylabel("Renewal Rate (%)")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(axis="x", rotation=rotation)
    ax.legend(fontsize=9)

bar_renewal(axes[0,0], "acquisition_channel",  "By Acquisition Channel", rotation=15)
bar_renewal(axes[0,1], "coverage_tier",         "By Coverage Tier",
            order=["Elite","Premium","Standard","Liability Only"])
bar_renewal(axes[0,2], "missed_payment_count",  "By Missed Payment Count")

bar_renewal(axes[1,0], "nsf_flag",              "NSF Flag")
bar_renewal(axes[1,1], "claim_severity_band",   "By Claim Severity",
            order=["None","Minor","Moderate","Major"])
bar_renewal(axes[1,2], "agency_type",            "By Agency Type")

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
num_cols = [
    "annual_premium", "tenure_months", "years_with_carrier",
    "missed_payment_count", "days_past_due_max", "payment_consistency",
    "nsf_flag", "late_pay_flag_3mo", "has_claim_12mo",
    "claim_count_12mo", "total_claim_amount", "days_since_last_claim",
    "multi_product_flag", "homeowner_flag", "prior_insurance_flag",
    "aqx_assisted_flag", "renewed",
]
corr = labeled_df[num_cols].replace(-1, np.nan).corr()["renewed"].drop("renewed").sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
colors = [BLUE if v > 0 else ORANGE for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color=NAVY, linewidth=0.8)
ax.set_title("Pearson Correlation with Renewed (Target)", fontsize=12)
ax.set_xlabel("Correlation Coefficient")
plt.tight_layout()
plt.show()

print("Top positive correlators (protective factors):")
print(corr.tail(5).round(3).to_string())
print()
print("Top negative correlators (risk factors):")
print(corr.head(5).round(3).to_string())

## 6. Key EDA Findings

**Renewal rate:** 74.8% overall across ~60,000 labeled policies (effective on or before 2024-12-31).

**Channel quality differential:** AQX channel achieves ~82% renewal vs. ~67% for Direct Web -- a 15 percentage point gap that drives the channel quality analysis on Tab 2 of the dashboard.

**Billing behavior is the strongest behavioral predictor:** Policies with 2+ missed payments have materially lower renewal rates than clean-billing policies. NSF events compound this effect.

**Claims do not uniformly reduce renewal:** Minor/Moderate severity claims show little impact on renewal. Major at-fault claims show the strongest negative effect, likely because the policyholder anticipates a premium increase.

**Multi-product attachment is protective:** Policyholders with both auto and renters policies have higher renewal rates, consistent with the cross-sell funnel analysis (Q10).

These findings directly motivate the feature selection in the gradient boosting model (Notebook 2).

---

# Part 2: Predictive Modeling

Gradient boosting lapse warning model, SHAP feature attribution, decile lift analysis, and scored dataset export.

# Waypoint Property & Casualty -- Predictive Modeling
**Project:** waypoint-retention-analytics
**Author:** Luciano Casillas
**Date:** 2026-05-02

Builds and evaluates a gradient boosting lapse warning model using XGBoost.
Produces SHAP feature importance, decile lift analysis, and a scored
dataset for the Streamlit dashboard (Tab 3: Predictive Model).

Leakage-prone columns excluded from training:
- `cltv_36mo` (derived from full 36-month outcome)
- `renewal_premium_change_pct` (known only after renewal offer)
- `post_renewal_coverage_tier` (known only after renewal decision)
- `outreach_contact_flag` (conditional leakage)
- `renewal_month_label`, `days_to_decision` (renewal outcome metadata)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import shap
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import OrdinalEncoder

# Locate project root
def find_project_root():
    p = Path.cwd()
    while p != p.parent:
        if (p / "PROJECT_MANIFEST.json").exists():
            return p
        p = p.parent
    return Path(__file__).resolve().parents[4]

PROJECT_ROOT = find_project_root()
DATA_PATH    = PROJECT_ROOT / "_build" / "workflows" / "02_data_generation" / "outputs" / "waypoint_retention.csv"
OUT_DIR      = PROJECT_ROOT / "_build" / "workflows" / "04_modeling" / "outputs"
MODEL_DIR    = PROJECT_ROOT / "_build" / "workflows" / "04_modeling" / "modeling"

BLUE = "#0077B3"; NAVY = "#003366"; ORANGE = "#E07B00"; STEEL = "#A0B4C8"; GREEN = "#217A3C"
plt.rcParams.update({"figure.dpi": 100, "font.size": 11})

print("Phase 4 Modeling Notebook")
print(f"XGBoost version: {xgb.__version__}")
print(f"SHAP version:    {shap.__version__}")

## 1. Load Data and Build Features

In [ ]:
df = pd.read_csv(DATA_PATH)
df["claim_severity_band"] = df["claim_severity_band"].fillna("None")
df["effective_date"] = pd.to_datetime(df["effective_date"])

# Filter to known renewal outcomes
labeled = df[df["renewed"].isin([0, 1])].copy()
print(f"Labeled policies: {len(labeled):,}  |  Renewal rate: {labeled['renewed'].mean():.1%}")

# Time-based split
cutoff = pd.Timestamp("2024-07-01")
train_df = labeled[labeled["effective_date"] < cutoff].copy()
test_df  = labeled[labeled["effective_date"] >= cutoff].copy()
print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")

y_train = train_df["renewed"]
y_test  = test_df["renewed"]

In [ ]:
# Feature engineering
LEAKAGE = ["cltv_36mo","renewal_premium_change_pct","post_renewal_coverage_tier",
            "outreach_contact_flag","renewal_month_label","days_to_decision"]
META    = ["customer_id","policy_id","effective_date","renewed"]
CAT     = ["age_band","state","metro_area","acquisition_channel","agency_type",
           "policy_type","coverage_tier","payment_method","billing_frequency",
           "claim_severity_band","cohort_quarter"]

def build_X(df, encoder=None, fit=False):
    X = df.drop(columns=[c for c in LEAKAGE + META if c in df.columns], errors="ignore")
    cats = [c for c in CAT if c in X.columns]
    if fit or encoder is None:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X[cats] = encoder.fit_transform(X[cats])
    else:
        X[cats] = encoder.transform(X[cats])
    return X.astype(float), encoder

X_train, enc = build_X(train_df, fit=True)
X_test,  _   = build_X(test_df, encoder=enc)
print(f"Features: {X_train.shape[1]}")
print(list(X_train.columns))

## 2. Train XGBoost Model

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.80, colsample_bytree=0.75,
    min_child_weight=15, gamma=1.0, reg_alpha=0.5, reg_lambda=2.0,
    objective="binary:logistic", eval_metric="auc",
    random_state=42, verbosity=0,
)
model.fit(X_train, y_train, verbose=False)

# Evaluate
prob_test = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, prob_test)
print(f"Test ROC-AUC: {auc:.4f}")
print()
print(classification_report(y_test, (prob_test >= 0.748).astype(int),
      target_names=["Lapsed", "Renewed"]))

## 3. SHAP Feature Importance

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(X_train), 5000, replace=False)
X_sample   = X_train.iloc[sample_idx]

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

mean_abs_shap = pd.DataFrame({
    "feature": X_train.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors = [BLUE if i == 0 else STEEL for i in range(len(mean_abs_shap))]
ax.barh(mean_abs_shap["feature"][::-1], mean_abs_shap["mean_abs_shap"][::-1], color=colors[::-1])
ax.set_title("SHAP Feature Importance (Mean |SHAP Value|)", fontsize=12)
ax.set_xlabel("Mean Absolute SHAP Value")
plt.tight_layout()
plt.show()

print("Top 10 features by SHAP:")
print(mean_abs_shap.head(10).to_string(index=False))

## 4. Decile Lift Analysis

In [ ]:
lapse_prob = 1.0 - prob_test
decile = pd.qcut(-lapse_prob, q=10, labels=list(range(1, 11))).astype(int)

lift_df = pd.DataFrame({
    "lapse_prob":  lapse_prob,
    "actual_lapse": (y_test.values == 0).astype(int),
    "decile": decile,
}).groupby("decile").agg(
    policy_count=("actual_lapse","count"),
    lapse_rate=("actual_lapse","mean"),
    avg_lapse_prob=("lapse_prob","mean"),
).reset_index()

book_lapse = (y_test == 0).mean()
lift_df["lift"] = (lift_df["lapse_rate"] / book_lapse).round(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Lift bar chart
colors = [BLUE if i < 2 else STEEL for i in range(10)]
axes[0].bar(lift_df["decile"], lift_df["lift"], color=colors)
axes[0].axhline(1.0, color=ORANGE, linestyle="--", label="Baseline (1.0x)")
axes[0].set_title("Lift by Lapse Propensity Decile (D1 = Highest Risk)")
axes[0].set_xlabel("Decile")
axes[0].set_ylabel("Lift vs. Baseline")
axes[0].set_xticks(range(1, 11))
axes[0].legend()

# Cumulative gain curve
cumulative = lift_df["lapse_rate"] * lift_df["policy_count"]
total_lapse = cumulative.sum()
pct_contacted = np.arange(1, 11) * 10
pct_captured  = cumulative.cumsum() / total_lapse * 100

axes[1].fill_between(pct_contacted, pct_captured, alpha=0.15, color=BLUE)
axes[1].plot(pct_contacted, pct_captured, color=BLUE, marker="o", linewidth=2, label="Model")
axes[1].plot([0, 100], [0, 100], color=STEEL, linestyle="--", label="Random baseline")
axes[1].set_title("Cumulative Gain Curve")
axes[1].set_xlabel("% of Policyholders Contacted")
axes[1].set_ylabel("% of Lapses Captured")
axes[1].legend()
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()
print(lift_df[["decile","lapse_rate","lift"]].to_string(index=False))

## 5. Confusion Matrix

In [ ]:
# Use renewal rate as threshold (predict lapse if prob_renewed < book_renewal_rate)
book_renewal = y_train.mean()
pred = (prob_test < book_renewal).astype(int)  # 1 = predicted lapse

y_lapse_true = (y_test == 0).astype(int)  # 1 = actually lapsed
cm = confusion_matrix(y_lapse_true, pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
cm_data = np.array([[tn, fp], [fn, tp]])
im = ax.imshow(cm_data, cmap="Blues")

labels = [["True Neg\n(Correctly\nflagged safe)", f"False Pos\n(Unnecessary\noutreach)"],
          ["False Neg\n(Missed\nlapse)", "True Pos\n(Correctly\nflagged risk)"]]

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_data[i,j]:,}\n{labels[i][j]}", ha="center", va="center",
                fontsize=9, color="white" if cm_data[i,j] > cm_data.max()*0.5 else "black")

ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted Safe", "Predicted At-Risk"])
ax.set_yticklabels(["Actually Safe", "Actually Lapsed"])
ax.set_title(f"Confusion Matrix (threshold={book_renewal:.0%})")
plt.tight_layout()
plt.show()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"At-risk precision (of those flagged, % truly lapsed): {precision:.1%}")
print(f"At-risk recall    (of all lapses, % flagged):         {recall:.1%}")
print(f"\nSo what: Contacting the top-risk decile reaches {lift_df.iloc[0]['lapse_rate']:.0%} "
      f"lapse rate vs. {book_lapse:.0%} book average -- {lift_df.iloc[0]['lift']:.1f}x lift.")

## 6. Score All Policies and Save Outputs

In [ ]:
# Score full 90K dataset
X_full, _ = build_X(df.copy(), encoder=enc, fit=False)
lapse_prob_full   = 1.0 - model.predict_proba(X_full)[:, 1]
decile_full       = pd.qcut(-lapse_prob_full, q=10, labels=list(range(1, 11))).astype(int)

scored = df.copy()
scored["lapse_prob"]    = lapse_prob_full.round(4)
scored["model_decile"]  = decile_full
scored["high_risk_flag"] = (decile_full <= 2).astype(int)

print(f"Scored: {len(scored):,} policies")
print(f"High-risk (deciles 1-2): {scored['high_risk_flag'].mean():.1%}")

# Save scored dataset
scored.to_csv(OUT_DIR / "waypoint_retention_scored.csv", index=False)
print(f"Saved: {OUT_DIR / 'waypoint_retention_scored.csv'}")

# Save SHAP values (all features, not just top 15)
full_shap = pd.DataFrame({
    "feature": X_train.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)
full_shap["mean_abs_shap"] = full_shap["mean_abs_shap"].round(5)
full_shap.to_csv(OUT_DIR / "shap_values.csv", index=False)
print(f"Saved: {OUT_DIR / 'shap_values.csv'}")

In [ ]:
# Save model metrics JSON
metrics = {
    "test_auc":      round(float(auc), 4),
    "test_accuracy": round(float(((prob_test >= 0.748) == y_test).mean()), 4),
    "high_risk_pct": round(float(scored["high_risk_flag"].mean()), 4),
    "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
    "decile_table":  lift_df.to_dict(orient="records"),
    "feature_count": int(X_train.shape[1]),
    "feature_names": list(X_train.columns),
    "top_5_features": full_shap["feature"].head(5).tolist(),
    "train_n": int(len(X_train)),
    "test_n":  int(len(X_test)),
}
with open(OUT_DIR / "model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)
print(f"Saved: {OUT_DIR / 'model_metrics.json'}")
print()
print("=== Phase 4 Complete ===")
print(f"Test AUC:         {metrics['test_auc']}")
print(f"D1 Lift:          {lift_df.iloc[0]['lift']:.2f}x")
print(f"High-risk pct:    {metrics['high_risk_pct']:.1%}")
print(f"Top feature:      {metrics['top_5_features'][0]}")